# Fase 3.3: Modelado - Modelo fundacional

In [1]:
# Importar dependencias.
import constants
import numpy as np
import torch
import warnings
from chronos import Chronos2Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from util import csv_util

# No mostrar mensajes de advertencia.
warnings.filterwarnings('ignore')

In [2]:
import chronos
print(chronos.__version__)

2.3.2


## 1. Introducción

Los modelos fundacionales han supuesto un cambio de paradigma para el análisis de las series temporales. A diferencia de los modelos clásicos de aprendizaje profundo, que se entrenan para un fin determinado, estos modelos se entrenan una sola vez con conjuntos masivos. Una vez validados, pueden utilizarse directamente para resolver diversos problemas de predicción.

## 2. Chronos-2

In [3]:
def load_dataset():
    df = csv_util.read('dataset/processed_data/dataset.csv', usecols=['date', 'city', 'section', 'roadway', 'direction', 'lane', 'total_stress'],
                        dtype={'city':str, 'section':int, 'roadway':int, 'direction':int, 'lane':int, 'total_stress':float},
                        date_formats=[('date', '%Y-%m-%d')])

    df[f'target_stress_{constants.HORIZON_DAYS}d'] = (df.groupby(['city', 'section', 'roadway', 'direction', 'lane'])['total_stress']
                                                        .rolling(window=constants.HORIZON_DAYS, min_periods=constants.HORIZON_DAYS)
                                                        .sum()
                                                        .shift(-constants.HORIZON_DAYS)
                                                        .reset_index(level=[0, 1, 2, 3, 4], drop=True))

    return df

def obtain_test(df):
    return df[df['date'].dt.year >= constants.TEST_BOUNDARY_YEAR]

def evaluate(valid):
    y_true = valid[f'target_stress_{constants.HORIZON_DAYS}d'].to_numpy(dtype=float)
    y_pred = valid[f'target_stress_{constants.HORIZON_DAYS}d_pred'].to_numpy(dtype=float)

    mae = mean_absolute_error(y_true, y_pred)
    mae_std = np.std(np.abs(y_true - y_pred))
    rmse = root_mean_squared_error(y_true, y_pred)
    mre = np.mean(np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), 1e-5))

    print(f'MAE: {mae:.4f} ± {mae_std:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'MRE: {mre:.4f}')

def predict(pipeline, df_test):
    target_col = f'target_stress_{constants.HORIZON_DAYS}d'
    pred_col = f'{target_col}_pred'
    keys_by_lane = ['city', 'section', 'roadway', 'direction', 'lane']
    df_eval = df_test.sort_values(keys_by_lane + ['date']).copy()
    df_eval[pred_col] = np.nan

    for _, group in df_eval.groupby(keys_by_lane, sort=False):
        values = group['total_stress'].astype(float).to_numpy()

        if len(values) < constants.WINDOW_SIZE + constants.HORIZON_DAYS - 1:
            continue

        for i in range(constants.WINDOW_SIZE, len(values) - constants.HORIZON_DAYS + 2):
            historical_window = values[i - constants.WINDOW_SIZE:i]
            max_val = np.max(np.abs(historical_window))
            normalized_window = historical_window / max_val if max_val > 0 else historical_window

            # Chronos 2.3.2 espera tensor 3D: (n_series, n_variates, seq_len).
            batch = torch.tensor([normalized_window], dtype=torch.float32).unsqueeze(1)
            if hasattr(pipeline, 'device'):
                batch = batch.to(pipeline.device)

            prediction = pipeline.predict(batch, prediction_length=constants.HORIZON_DAYS)

            # En Chronos 2.3.2, predict() devuelve list[torch.Tensor].
            if isinstance(prediction, list):
                pred_tensor = prediction[0]
                if hasattr(pred_tensor, 'detach'):
                    pred_tensor = pred_tensor.detach().cpu().numpy()
                pred_array = np.asarray(pred_tensor, dtype=float)
            else:
                pred_array = np.asarray(prediction, dtype=float)

            if pred_array.ndim == 3:
                pred_vector = np.median(pred_array, axis=1).reshape(-1)
            elif pred_array.ndim == 2:
                pred_vector = pred_array[0] if pred_array.shape[0] == 1 else np.median(pred_array, axis=0)
            elif pred_array.ndim == 1:
                pred_vector = pred_array
            else:
                pred_vector = pred_array.reshape(-1)[:constants.HORIZON_DAYS]

            pred_vector = np.asarray(pred_vector, dtype=float)[:constants.HORIZON_DAYS]
            if max_val > 0:
                pred_vector = pred_vector * max_val

            future_total = float(np.sum(pred_vector))
            row_idx = group.index[i - 1]
            df_eval.loc[row_idx, pred_col] = future_total

    valid = df_eval.dropna(subset=[target_col, pred_col]).copy()
    if valid.empty:
        print('No se pudieron alinear suficientes series para la evaluación.')
    else:
        evaluate(valid)

    return df_eval

def save_results(df_test, model_name):
    csv_util.save(df_test, f'model/{model_name}/result/{model_name}_results.csv', mode='w')

In [4]:
print('Chronos-2')
print('1. Obtener conjunto de datos de pruebas')
df_test = obtain_test(load_dataset())

print('2. Cargar modelo preentrenado')
pipeline = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cuda', dtype=torch.float32)

print('3. Predecir y evaluar')
df_test = predict(pipeline, df_test)

print('4. Guardar predicciones')
save_results(df_test, 'Chronos-2')

print('FIN Chronos-2')

Chronos-2
1. Obtener conjunto de datos de pruebas
2. Cargar modelo preentrenado
3. Predecir y evaluar
MAE: 135126.4068 ± 177869.5814
RMSE: 223375.7682
MRE: 0.1782
4. Guardar predicciones
FIN Chronos-2
